# ML-10 — Content Action Playbook

> **Author:** Ahmed El-Dasoky · FlyRank ML Internship · Applied Search Intelligence track
> **Lane:** Refresh / Content Opportunity Scoring
> **Data:** bundled `data/raw/content_refresh_anonymized.csv` (30,000 pseudonymized pages, 44 columns) + the reference pipeline in `scripts/` and its committed outputs.

Run top to bottom — every code cell reads only public, bundled data and prints numbers, never raw queries or client names.


# ML-10 — Content Action Playbook


## 1. Ranked actions + reason codes

The queue maps every page to an action a human can execute: **refresh** (declining + visible + low CTR), **review_ctr** (visible + low CTR), **monitor** (healthy or tiny), with reason codes from the baseline. Read top-down: bigger pages first (higher impressions), regardless of what a model says.

In [ ]:
import pandas as pd, os
os.makedirs("work/outputs", exist_ok=True)
q = pd.read_csv("work/outputs/baseline_action_score.csv")
print(q["action"].value_counts().to_string())
print(q.head(10).to_string(index=False))

## 2. Intended use and limits
- **Who uses it:** a content operator triaging a weekly refresh queue.
- **For what:** prioritising *which page to read next*; the action is a hypothesis to confirm, not an instruction to execute blindly.
- **Where it stops being valid:** out of the 90-day window, beyond the bundled client distribution, or as a *prediction of recovery*.

## 3. Human review + the no-go list
- A human must confirm: demand still exists (`search_volume`, competition), the page is actually stale/non-performing on inspection, and no manual priority overrides.
- **Never automate:** auto-refreshing content without review; touching pages where demand collapsed; or using the model to delete/adjudicate clients.

## 4. Monitoring / retrain triggers
- Retrain when: the decline-rate base rate drifts (e.g., label balance moves by >±0.1), a new data release arrives, or Precision@50 on a fresh holdout drops >0.05.
- Re-tune the threshold only on a fresh grouped split, never on the same holdout.

## 5. Exports for the paper

Write the queue and a couple of charts to `work/outputs/` for the capstone paper + deployed page.

In [ ]:
import pandas as pd, matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt, os
os.makedirs("work/outputs", exist_ok=True)

q = pd.read_csv("work/outputs/baseline_action_score.csv")
fig, ax = plt.subplots(figsize=(6,3))
q["score"].hist(bins=40, ax=ax); ax.set_title("Baseline action-score distribution")
plt.tight_layout(); plt.savefig("work/outputs/playbook_score_dist.png", dpi=110)

q.to_csv("work/outputs/action_playbook.csv", index=False)
print("exports written to work/outputs/")

## Self-check

- [x] Ranked actions with reason codes
- [x] Intended use + limits
- [x] Human review + no-go list
- [x] Monitoring triggers
- [x] Paper exports produced